## Avaliação automática de respostas utilizando LLM as a Judge

Neste laboratório, implementaremos um mecanismo simples de avaliação automática de respostas geradas por LLMs, utilizando outro modelo de linguagem como avaliador, técnica conhecida como LLM as a Judge.

A ideia é utilizar um segundo modelo para analisar a qualidade de uma resposta gerada, atribuindo uma pontuação baseada em critérios como correção, clareza e completude. Esse tipo de abordagem é frequentemente utilizado para avaliação automática de sistemas de IA generativa.

Além disso, integraremos o processo de avaliação com Langfuse, permitindo registrar cada execução e acompanhar os resultados das avaliações ao longo do tempo.

### Objetivo didático

Demonstrar na prática:
- utilizar LLMs para avaliar respostas geradas por outros modelos
- criar prompts estruturados para avaliação automática
- gerar pontuações de qualidade para respostas de IA
- estruturar resultados em formato JSON para análise
- integrar processos de avaliação com observabilidade usando Langfuse
- compreender como técnicas de LLM as a Judge podem apoiar avaliação de sistemas de IA

In [1]:
%%capture
! pip install openai

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

# Credencial embedding e LLM
api_key = os.getenv("OCI_API_KEY")
base_url = "https://inference.generativeai.us-chicago-1.oci.oraclecloud.com/20231130/actions/v1"  # ou sua URL


### Parte 1 - Implementação simples

### Sistema que se quer avaliar

In [4]:
from openai import OpenAI

client = OpenAI(base_url=base_url, api_key=api_key)
question = "o que são guardrails?"
response = client.chat.completions.create(
    model="xai.grok-4-fast-non-reasoning",
    messages=[
        {"role": "user", "content": question}
    ]
)

answer = response.choices[0].message.content

### LLM as judge 

In [7]:
judge_prompt = f"""
You are an AI evaluator.

Question:
{question}

Answer:
{answer}

Evaluate the answer from 1 to 10 considering:
- correctness
- clarity
- completeness

Return JSON:
{{
"score": number,
"reason": "explanation"
}}

final answer always must be in pt-br
"""

judge_response = client.chat.completions.create(
    model="xai.grok-4-fast-non-reasoning",
    messages=[
        {"role": "user", "content": judge_prompt}
    ]
)

print("\033[94mJUDGE:\033[0m")
print(judge_response.choices[0].message.content)

JUDGE:
{
  "score": 9,
  "reason": "A resposta é correta, pois explica com precisão os significados de 'guardrails' em contextos literais e metafóricos, apoiando-se em evidências reais como normas de segurança e práticas de IA. É clara, com estrutura organizada em seções e linguagem acessível. É completa, cobrindo usos principais (engenharia, IA, negócios) e convidando a mais detalhes, embora possa expandir ligeiramente exemplos em contextos adicionais para perfeição absoluta."
}


### Parte 2 - Adicionando a observabilidade

In [19]:
from langfuse.openai import OpenAI
from langfuse import Langfuse
import uuid

trace_id = str(uuid.uuid4()) # Identificação unica de cada execução
langfuse = Langfuse()

client = OpenAI(base_url=base_url, api_key=api_key)

question = "Agent Intelligence"

# geração da resposta
response = client.chat.completions.create(
    model="openai.gpt-oss-20b",
    messages=[{"role":"user","content":question}],
    metadata={
        "user_id": "lab-user",
        "experiment": "llm-lab"
    },
    extra_headers={
        "x-langfuse-trace-id": trace_id
    }
)

answer = response.choices[0].message.content


# LLM-as-a-Judge
judge_prompt = f"""
Score the answer from 1-10.

Question: {question}
Answer: {answer}

Return only a number.
"""

judge = client.chat.completions.create(
    model="xai.grok-4-fast-non-reasoning",
    messages=[{"role":"user","content":judge_prompt}],
    metadata={
        "user_id": "lab-user",
        "experiment": "judge-llm-lab"
    },
    extra_headers={
        "x-langfuse-trace-id": trace_id
    }
)

score = float(judge.choices[0].message.content)


# registrar avaliação no Langfuse
langfuse.create_score(
    trace_id=trace_id,
    name="quality",
    value=score,
    comment="LLM-as-a-judge evaluation"
)

print(answer)

KeyboardInterrupt: 